## All output files have net length in METERS

# Driving

In [18]:
import pandas as pd

df = pd.read_csv("C:/Users/15177459/Desktop/gis_outputs/cost_od_matrix/transport/input/netlen_driving.csv")
print(df.columns)

Index(['FacilityID', 'SourceID', 'SourceOID', 'FromPosition', 'ToPosition',
       'FromCumul_DrvingTime', 'FromCumul_WalkTime', 'FromCumul_Length',
       'FromCumul_BikeTime', 'ToCumul_DrvingTime', 'ToCumul_WalkTime',
       'ToCumul_Length', 'ToCumul_BikeTime', 'Shape_Length'],
      dtype='str')


In [19]:
cross_5 = df[(df["ToCumul_DrvingTime"] < 5) & (df['ToCumul_DrvingTime'] > 5)] 
cross_10 = df[(df["ToCumul_DrvingTime"] < 10) & (df['ToCumul_DrvingTime'] > 10)] 

print("Rows crossing 5:", len(cross_5))
print("Rows crossing 10:", len(cross_10))

Rows crossing 5: 0
Rows crossing 10: 0


In [20]:
print(df["ToCumul_DrvingTime"].max())
# Check the sum of length for ONLY the rows between 5 and 10
gap_sum = df[(df["ToCumul_DrvingTime"] > 5) & (df["ToCumul_DrvingTime"] <= 10)]["Shape_Length"].sum()
print(f"Total length of segments between 5 and 10 mins: {gap_sum}")

10.0
Total length of segments between 5 and 10 mins: 111566.95325622581


In [21]:


# 2. Create the two sums separately
# Use different variable names for the masks to be 100% sure
mask_5 = df["ToCumul_DrvingTime"] <= 5
mask_10 = df["ToCumul_DrvingTime"] <= 10

d5 = df[mask_5].groupby("FacilityID")["Shape_Length"].sum().rename("netlen_5")
d10 = df[mask_10].groupby("FacilityID")["Shape_Length"].sum().rename("netlen_10")

# 3. Combine them using a simple concat (cleaner than a merge for this)
summary = pd.concat([d5, d10], axis=1).fillna(0).reset_index()

# 4. PROOF: Add a difference column
summary["diff"] = summary["netlen_10"] - summary["netlen_5"]

# 5. Check the results
print("First few rows of summary:")
print(summary.head())

print(f"\nTotal sum of netlen_5: {summary['netlen_5'].sum()}")
print(f"Total sum of netlen_10: {summary['netlen_10'].sum()}")
print(f"Number of rows where 10 is greater than 5: {(summary['diff'] > 0).sum()}")

# 6. Save with a new name
summary.to_csv("C:/Users/15177459/Desktop/gis_outputs/cost_od_matrix/transport/output/FINAL_CHECK.csv", index=False)

First few rows of summary:
   FacilityID     netlen_5    netlen_10  diff
0           1  1145.057381  1145.057381   0.0
1           2  2372.935249  2372.935249   0.0
2           3  1998.300241  1998.300241   0.0
3           4  2824.254334  2824.254334   0.0
4           5  3399.260503  3399.260503   0.0

Total sum of netlen_5: 2795535.589592117
Total sum of netlen_10: 2907102.5428483426
Number of rows where 10 is greater than 5: 29


# Walking

In [13]:
df = pd.read_csv("C:/Users/15177459/Desktop/gis_outputs/cost_od_matrix/transport/input/netlen_walking.csv")
print(df.columns)

Index(['FacilityID', 'SourceID', 'SourceOID', 'FromPosition', 'ToPosition',
       'FromCumul_DrvingTime', 'FromCumul_WalkTime', 'FromCumul_Length',
       'FromCumul_BikeTime', 'ToCumul_DrvingTime', 'ToCumul_WalkTime',
       'ToCumul_Length', 'ToCumul_BikeTime', 'Shape_Length'],
      dtype='str')


In [ ]:
cross_5 = df[(df["ToCumul_WalkTime"] < 5) & (df['ToCumul_WalkTime'] > 5)] 
cross_10 = df[(df["ToCumul_WalkTime"] < 10) & (df['ToCumul_WalkTime'] > 10)] 

print("Rows crossing 5:", len(cross_5))
print("Rows crossing 10:", len(cross_10))

Rows crossing 5: 3
Rows crossing 10: 0


In [ ]:
walk_5 = (
    df[df["ToCumul_WalkTime"] <= 5]
    .groupby("FacilityID", as_index=False)
    .agg(walk_5_netlen=("Shape_Length", "sum"))
    )

walk_10 = (
    df[df["ToCumul_WalkTime"] <= 10]
    .groupby("FacilityID", as_index=False)
    .agg(walk_10_netlen=("Shape_Length", "sum"))
    )

summary  = walk_5.merge(walk_10, on="FacilityID", how="outer").fillna(0)

summary.to_csv("C:/Users/15177459/Desktop/gis_outputs/cost_od_matrix/transport/output/netlen_walking_summary.csv")

# Cycling

In [16]:
df = pd.read_csv("C:/Users/15177459/Desktop/gis_outputs/cost_od_matrix/transport/input/netlen_cycling.csv")
print(df.columns)

Index(['FacilityID', 'SourceID', 'SourceOID', 'FromPosition', 'ToPosition',
       'FromCumul_DrvingTime', 'FromCumul_WalkTime', 'FromCumul_Length',
       'FromCumul_BikeTime', 'ToCumul_DrvingTime', 'ToCumul_WalkTime',
       'ToCumul_Length', 'ToCumul_BikeTime', 'Shape_Length'],
      dtype='str')


In [ ]:
cross_5 = df[(df["ToCumul_BikeTime"] < 5) & (df['ToCumul_BikeTime'] > 5)] 
cross_10 = df[(df["ToCumul_BikeTime"] < 10) & (df['ToCumul_BikeTime'] > 10)] 

print("Rows crossing 5:", len(cross_5))
print("Rows crossing 10:", len(cross_10))

Rows crossing 5: 3
Rows crossing 10: 0


In [ ]:
cycle_5 = (
    df[df["ToCumul_BikeTime"] <= 5]
    .groupby("FacilityID", as_index=False)
    .agg(cycle_5_netlen=("Shape_Length", "sum"))
    )

cycle_10 = (
    df[df["ToCumul_BikeTime"] <= 10]
    .groupby("FacilityID", as_index=False)
    .agg(cycle_10_netlen=("Shape_Length", "sum"))
    )

summary  = cycle_5.merge(cycle_10, on="FacilityID", how="outer").fillna(0)

summary.to_csv("C:/Users/15177459/Desktop/gis_outputs/cost_od_matrix/transport/output/netlen_cycling_summary.csv")